# Purpose 
 - "Is each individual record biologically and clinically plausible on it's own"

 This notebook applies predefined clinical rules to each individual record in the maternal health dataset.
Validation is performed at the record level, with biologically impossible values flagged as errors and clinically unlikely values flagged as warnings.
No records are removed or corrected in this step.

- Now we can touch df


### Clinical rules (imported from notebook2 )

In [28]:
clinical_rules = {  
    "age_range" : {
        "min":10,
        "max":55,
        "severity" : "error",
        "description" : "Maternal age outside reproductive age"
    }, 
    "bp_range": {
        "systolic_min": 70,
        "systolic_max": 250,
        "diastolic_min": 40,
        "diastolic_max": 150,
        "severity" : "error",
        "description" : "Blood pressure values outside physiological range"
    }, 
    "bp_logic" : {
        "rule": "systolic >= diastolic",
        "severity" : "error",   
        "description" : "Physiologically impossible blood pressure"
    },
    "risk_mismatch" : {
        "rule" : "high_bp and low_risk",
        "severity" : "warning",
        "description": "Potential risk misclassification"
    }
}

In [29]:
clinical_rules.keys()

dict_keys(['age_range', 'bp_range', 'bp_logic', 'risk_mismatch'])

In [30]:
import pandas as pd
import numpy as np

df = pd.read_csv ("D:\maternal-health-risk-analysis\maternal_health.csv" )
df.head()
df.shape


<>:4: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<>:4: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
C:\Users\khush\AppData\Local\Temp\ipykernel_12176\3479748404.py:4: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
  df = pd.read_csv ("D:\maternal-health-risk-analysis\maternal_health.csv" )


(1014, 7)

In [31]:
df["is_error"] = False
df["is_warning"] = False
df["error_reason"] = ""
df["warning_reason"] = ""


In [32]:
df.columns

Index(['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate',
       'RiskLevel', 'is_error', 'is_warning', 'error_reason',
       'warning_reason'],
      dtype='object')

In [33]:
def apply_bp_logic_rule(df):
    mask = df["SystolicBP"] < df["DiastolicBP"]
    df.loc[mask, "is_error"] = True
    df.loc[mask, "error_reason"] += "Systolic BP lower than Diastolic BP; "
    return df

df = apply_bp_logic_rule(df)

# Age Rule
 

In [ ]:
age_rule = clinical_rules["age_range"]

mask = (df["Age"] < age_rule["min"]) | (df["Age"] > age_rule["max"])

df.loc[mask, "is_error"] = True
df.loc[mask, "error_reason"] += "Age outside biological range; "


In [35]:
assert "error_reason" in df.columns
assert "warning_reason" in df.columns



In [ ]:
df["is_error"].sum()

np.int64(45)

This denotes that within these records 45 records voilated at least one error level value !


How many records are there ?



In [37]:
len(df)

1014

Error % = (45/total) * 100 <br>
 4.43 %

 

# Which rule is dominating ?


In [38]:
df[df["is_error"]]["error_reason"].value_counts()

error_reason
Age outside biological range;     45
Name: count, dtype: int64

In [39]:
df[df["is_error"]].head()

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel,is_error,is_warning,error_reason,warning_reason
36,60,120,80,6.1,98.0,75,low risk,True,False,Age outside biological range;,
54,60,90,65,7.0,98.0,77,low risk,True,False,Age outside biological range;,
91,60,120,85,15.0,98.0,60,mid risk,True,False,Age outside biological range;,
99,60,90,65,6.8,98.0,77,mid risk,True,False,Age outside biological range;,
114,63,140,90,15.0,98.0,90,high risk,True,False,Age outside biological range;,


In [40]:
df[["is_error", "is_warning"]].value_counts()

is_error  is_warning
False     False         969
True      False          45
Name: count, dtype: int64

# Warning count 


In [41]:
warning_count = df["is_warning"].sum()
warning_pct = round((warning_count/len(df)) * 100, 2) 

warning_count , warning_pct

(np.int64(0), np.float64(0.0))

In [42]:
df[["is_error", "is_warning"]].value_counts()

is_error  is_warning
False     False         969
True      False          45
Name: count, dtype: int64

Out of 1014 records, approx 4.4 % were flagged with error level violations idicating biologicallu implausible values.


In [43]:
df["is_warning"].sum()

np.int64(0)

In [44]:
df["RiskLevel"].value_counts()

RiskLevel
low risk     406
mid risk     336
high risk    272
Name: count, dtype: int64

In [45]:
(
    (df["SystolicBP"] >= 160) &
     (df["DiastolicBP"] >= 110)
).sum()

np.int64(0)

In [47]:
df[df["SystolicBP"] >= 160] [["SystolicBP", "DiastolicBP", "RiskLevel"]].head()

,SystolicBP,DiastolicBP,RiskLevel
123,160,100,high risk
130,160,100,high risk
166,160,100,high risk
262,160,100,high risk
362,160,100,high risk


In [51]:
def apply_risk_mismatch_rule(df):
    mask = (
        (df["SystolicBP"] >= 160) &
        (df["DiastolicBP"] >= 110) &
        (df["RiskLevel"].str.strip().str.lower() == "low risk")
    )
    df.loc[mask, "is_warning"] = True
    df.loc[mask, "warning_reason"] += "Severe BP with Low Risk label; "
    return df

df = apply_risk_mismatch_rule(df)

In [52]:
df["is_warning"].sum()


np.int64(0)

In [50]:
df["RiskLevel"].value_counts()

RiskLevel
low risk     406
mid risk     336
high risk    272
Name: count, dtype: int64

In [54]:
(
    (df["SystolicBP"] >= 160) &
    (df["DiastolicBP"] >= 110)
).sum()

np.int64(0)

No records met the criteria for warning-level risk mismatch between severe blood pressure values and low-risk labels.
This suggests that the dataset’s risk categorization is internally consistent with respect to extreme vital sign thresholds.